# We will do all our analysis here

In [5]:
import polars as pl

from pathlib import Path

from circadian_cell_painting import (
    CosinorVisualize,
    OscillationResults,
    cosinor_fitted_long,
    detect,
    CosinorDetection
)
# from circadian_cell_painting.osc_detection import detect_cosinor_multiperiod

rows = [chr(r) for r in range(ord('A'), ord('P') + 1)]
cols = [f'{c:02d}' for c in range(1, 25)]
all_wells = pl.DataFrame({
    '384_Well': [f'{r}{c}' for r in rows for c in cols]
})

# display(all_wells)

raw_dataset = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                   "Study/analysis_ready_data/2026-05-11_Harmony_Results"
                   "._.DATASET1.tsv")

platemap_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Liv"
                     "e Study/platemaps/2026_05_26plate_map.xlsx")

platemap = (
    pl.read_excel(platemap_path, engine='openpyxl')
    .select(['384_Well', 'MoleculeID', 'Human_Readable_Name'])
    .join(all_wells, on="384_Well", how="right")
    .with_columns(
        pl.concat_str([
            pl.col('MoleculeID').cast(str),
            pl.col('Human_Readable_Name').cast(str)
        ], separator=' | ').alias('group_label')
    )
    .with_columns(
        pl.col('group_label').fill_null('nan | nan')
    )
    .rename({"384_Well": "WellName"})
    )

display(platemap)

MoleculeID,Human_Readable_Name,WellName,group_label
str,str,str,str
null,null,"""A01""","""nan | nan"""
null,null,"""A02""","""nan | nan"""
null,null,"""A03""","""nan | nan"""
null,null,"""A04""","""nan | nan"""
null,null,"""A05""","""nan | nan"""
…,…,…,…
null,null,"""P20""","""nan | nan"""
null,null,"""P21""","""nan | nan"""
null,null,"""P22""","""nan | nan"""


## Preprocessing

Based on the gene expression data, we can make an
informed decision to set our window to be 30 hours.

If this is not enough timepoints, we can set this down to a 24 hour window

In [6]:
# I want to try this but with a 24 hour window
window_hours = 24
sampling_interval = 1.5 # 1.5 hours
n_timepoints = round(window_hours / sampling_interval)

print(n_timepoints)

out_file = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                "Study/preprocessed_data")
# out_file = out_file / "2026-05-11_Harmony_Results._.DATASET1.parquet"
out_file = out_file / "2026-05-11_Harmony_Results._.DATASET1-24hrWindow.parquet"

!ccellpaint preprocess \
    "{raw_dataset}" \
    "{out_file}" \
    -w WellName -t Timepoint --window {n_timepoints}

16
INFO:root:3786 features -> 16 batches of 250
Batching progress: 100%|████████████████████████| 16/16 [19:37<00:00, 73.59s/it]
INFO:root:denoise/detrend: window size set to: 17 timepoints
INFO:root:denoise/detrend: completed 2 pass rolling avg
INFO:root:denoise/detrend: subtracted raw signal from noise


## Analysis and review

In [7]:
# preprocessed_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
#                          "Study/preprocessed_data/2026-05-11_Harmony_Results._"
#                          ".DATASET1.parquet")

preprocessed_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                         "Study/preprocessed_data/2026-05-11_Harmony_Results._"
                         ".DATASET1-24hrWindow.parquet")

dataset = pl.scan_parquet(preprocessed_path)

tpoints = (dataset.
        select('Timepoint').
        unique().
        collect().
        to_series().
        to_list())

# print(*sorted(tpoints), sep="\n") 
print(f"Number of time points: {len(tpoints)}")

print(*list(dataset.collect_schema().to_python()), sep="\n")

Number of time points: 29
WellName
Timepoint
feature
mean
median
stddev
q0
q10
q25
q50
q75
q90
q100


In [8]:
# analysis

period = 30
time_offset = 6
well_col = "WellName"
feature_col = "feature"
timepoint_col = "Timepoint"
value_col = "median"

results_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                    "Rhythms/Live Study/results")
results_file = results_path / "2026-05-11_Harmony_Results._.DATASET1._.RESULTS.parquet"

correction_group = ['group_label']

model = CosinorDetection(
    period=30,
    intervals=1,
    time_offset=time_offset,
)

results: OscillationResults = detect(
    file_path=preprocessed_path,
    platemap_path=platemap,
    out_file=results_file,
    well_column=well_col,
    feature_col=feature_col,
    timepoint_col=timepoint_col,
    value_col=value_col,
    stat=value_col,
    correction_group=correction_group,
    detector=model
)

pickle_out = Path("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/pickles")
# pickle_file = pickle_out / "2026-05-11_Harmony_Results._.DATASET1._.RESULTS_OBJ.pickle"
pickle_file = pickle_out / "2026-05-11_Harmony_Results._.DATASET1._.RESULTS_OBJ-24hrWindow.pickle"

results.to_pickle(pickle_file)

Well progress: 100%|██████████| 28/28 [05:42<00:00, 12.25s/it]


## Visualization

In [9]:
pickle_out = Path("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/pickles")
# pickle_file = pickle_out / "2026-05-11_Harmony_Results._.DATASET1._.RESULTS_OBJ.pickle"
pickle_file = pickle_out / "2026-05-11_Harmony_Results._.DATASET1._.RESULTS_OBJ-24hrWindow.pickle"


results = OscillationResults.load_pickle(pickle_file)

print(results)

OscillationResults(well_col='WellName', feature_col='feature', timepoint_col='Timepoint', value_col='median', stat='median', detector_params={'period': 30, 'time_interval': 1, 'time_offset': 6, 'start_index': 0}, results_path=PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/results/2026-05-11_Harmony_Results._.DATASET1._.RESULTS.parquet'), data_path=PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data/2026-05-11_Harmony_Results._.DATASET1-24hrWindow.parquet'), platemap=None, detector_name='CosinorFitting', adj_alpha=0.05, adj_method='fdr_bh', correction_group=['group_label'], start_time=None, end_time=None)


In [11]:
dataset_name = results.data_path.stem
visual_out = Path("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/visuals")
# visual_out = visual_out / f"{dataset_name}._.COSINOR-RES"
visual_out = visual_out / f"{dataset_name}._.COSINOR-RES"

print(visual_out)

visualize = CosinorVisualize(results, tick_interval=6)
visualize.view(visual_out)

/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/visuals/2026-05-11_Harmony_Results._.DATASET1-24hrWindow._.COSINOR-RES


well_processing: 100%|██████████| 27/27 [04:23<00:00,  9.76s/it]
